# 06. Contextual Compression (문맥 압축)

## 실습 목적

이 실습에서는 **AI 트렌드 보고서 Markdown 기반 Vector Store**를 사용하여 Contextual Compression을 적용한다.

검색된 청크 전체를 LLM에 전달하는 대신,  
**질문과 관련된 부분만 추출하거나 관련 없는 문서를 제거**하여 컨텍스트 노이즈를 줄이는 것이 목표이다.

```text
검색된 청크
┌─────────────────────────────────────────┐
│ 관련 없는 내용                            │
│ 질문과 관련 있는 핵심 내용  ← 이 부분만 사용 │ 
└─────────────────────────────────────────┘

압축 후 context
  ↓
질문과 관련 있는 내용 중심으로 답변 생성
```

## 주요 Compressor

| 방법 | 설명 |
|---|---|
| LLMChainExtractor | LLM으로 질문과 관련된 문장만 추출 |
| LLMChainFilter | 질문과 관련 없는 문서 자체를 제거 |
| EmbeddingsFilter | 임베딩 유사도 기준으로 관련 문서 필터링 |
| Pipeline | Reranking과 Compression을 단계적으로 결합 |

## Reranking과 Contextual Compression의 차이

Reranking은 검색된 후보 문서의 순서를 다시 정렬하고, 상위 문서를 선택하는 방법이다.  
즉, “어떤 문서를 사용할 것인가”를 결정한다.

Contextual Compression은 선택된 문서 안에서 질문과 관련 있는 부분만 추출하거나, 관련성이 낮은 문서를 제거하는 방법이다.  
즉, “선택된 문서에서 어떤 부분을 사용할 것인가”를 결정한다.

```text
Reranking
→ 문서 단위 선택과 정렬

Contextual Compression
→ 문서 내부 내용 추출 또는 문서 필터링
```

## 사용 데이터

- 이전 실습에서 생성한 AI 트렌드 보고서 Markdown 기반 ChromaDB를 사용한다.
- 먼저 `02_data_pdf_to_md.ipynb`를 실행하여 아래 Vector Store가 생성되어 있어야 한다.

```text
collection_name = ai_trend_report_2026_md
persist_directory = chroma_db/ai_trend_report_md
```

In [6]:
from pathlib import Path
from dotenv import load_dotenv
load_dotenv(override=True, dotenv_path="../.env")

from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_chroma import Chroma

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

# 02_data_pdf_to_md.ipynb에서 생성한 AI 트렌드 보고서 Markdown 기반 ChromaDB 사용

DB_PATH = Path("chroma_db/ai_trend_report_md")
COLLECTION_NAME = "ai_trend_report_2026_md"
vector_store = Chroma(
    collection_name=COLLECTION_NAME,
    embedding_function=embeddings,
    persist_directory=DB_PATH
)

base_retriever = vector_store.as_retriever(search_kwargs={"k": 5})

print("AI 트렌드 보고서 Vector Store 준비 완료")
print(f"저장된 청크 수: {vector_store._collection.count()}")

AI 트렌드 보고서 Vector Store 준비 완료
저장된 청크 수: 29


## 문서 출력 함수

In [7]:
def doc_label(doc):
    metadata = doc.metadata
    source = metadata.get("source", "?")
    chunk_id = metadata.get("chunk_id", "?")
    doc_type = metadata.get("doc_type", "?")
    source_name = metadata.get("source_name", "?")

    return f"chunk_id={chunk_id} | doc_type={doc_type} | source_name={source_name} | source={source}"

## LLMChainExtractor - 관련 부분만 추출

`LLMChainExtractor`는 검색된 문서 전체를 그대로 사용하지 않고,  
질문과 관련된 문장 또는 구절만 LLM이 추출하도록 한다.

AI 트렌드 보고서처럼 하나의 청크 안에 여러 주제가 섞여 있는 경우,  
질문과 무관한 문장을 줄여 답변 품질을 높이는 데 도움이 된다.

In [8]:
from langchain_classic.retrievers.document_compressors import LLMChainExtractor
from langchain_classic.retrievers import ContextualCompressionRetriever

extractor = LLMChainExtractor.from_llm(llm)

extractor_retriever = ContextualCompressionRetriever(
    base_compressor=extractor,
    base_retriever=base_retriever
)

question = "AI 에이전트가 기업 업무 방식에 어떤 변화를 가져올 것으로 전망되나요?"

# 압축 전
raw_docs = base_retriever.invoke(question)

print("[압축 전] 전체 청크")
for i, doc in enumerate(raw_docs[:2], start=1):
    source = doc.metadata.get("source", "?")
    print(f"\n[{i}] {len(doc.page_content)}자 | source={source}")
    print(doc.page_content[:500])

# 압축 후
compressed_docs = extractor_retriever.invoke(question)

print("\n\n[압축 후] 질문과 관련된 부분만 추출")
for i, doc in enumerate(compressed_docs, start=1):
    print(f"\n[{i}] {len(doc.page_content)}자 | {doc_label(doc)}")
    print(doc.page_content[:500])

[압축 전] 전체 청크

[1] 784자 | source=data\AI@Data_Report_CLEANED.md
- - 이와 함께 기존 산업의 가치 사슬도 전면 재편이라기보다는 핵심 공정·고부가 서비스 단에서 데이터·AI 중심 구조로 이동하는 압력이 강화되는 양상을 보일 것으로 전망됨 

- 기업 내부에서 AI 에이전트를 활용한 문서 처리, 고객 지원, 운영 자동화 등이 증가하며 – – 

- 사람 에이전트 시스템이 혼합된 업무 구조 가 일부 영역에서 확산될 가능성이 있음 

- - 기업 간(B2B) 시스템 연동을 통해 제한된 범위에서 AI 간 협업 형태의 자동화 사례도 등장할 것으로 예상되지만, 그 영향은 아직 초기 단계에 머물며 중장기적 구조 변화의 단초를 형성하는 수준으로 평가됨 

- ※ IoT 센서와 AI 기술을 통해 물류 및 공급망 전 과정을 자동화하고 최적화[4)] 

## **[ 그림 5 ] 주요 산업 분야별 AI 적용 사례** 

## **4.3 2026년 AI 기술 전망** 

- 2026년 AI 기술은 지능 구조 고도화 와 데이터 한계 보완을 중심 으로 진화할 전망임 

-

[2] 775자 | source=data\AI@Data_Report_CLEANED.md
- - 생성형 AI 고도화, AI 인프라 경쟁, 안전·책임성 강화, 국제 기준 정비 등 주요 변화 요인을 식별하여 산업·기술·정책 분야의 변화 방향성을 명확히 도출함 

- 더 나아가 분석 결과를 기반으로 국가 정책, 기업 전략, 기술 개발 방향 설정에 활용 가능한 실질적 인사이트를 제공하고, 중장기 전략 수립을 위한 근거를 제공하고자 함 

- - 특히 정책 담당자에게는 AI 기본법·규제체계 설계 등 법적 기반 마련 시 고려해야 할 우선순위를, 기업에는 투자·사업 포트폴리오 조정 방향을, 연구기관에는 후속 정량·정성 연구가 필요한 공백 영역을 제시하는 것을 목표로 함 

- 산업·기술·정책 세부 분석을 통해 다음과 같은 분야별 기대효과를 도출하고자 함 

- - 산업 분야에서는 A

## LLMChainFilter - 관련 없는 문서 제거

`LLMChainFilter`는 검색된 문서가 질문과 관련 있는지 LLM이 판단하게 한다.  
관련성이 낮은 문서는 context에 포함하지 않는다.

즉, 청크 내부를 잘라내기보다는 **문서 단위로 유지 또는 제거**하는 방식이다.

In [9]:
from langchain_classic.retrievers.document_compressors import LLMChainFilter

filter_compressor = LLMChainFilter.from_llm(llm)

filter_retriever = ContextualCompressionRetriever(
    base_compressor=filter_compressor,
    base_retriever=base_retriever
)

question = "AI 정책에서 안전성과 책임이 왜 중요한 이슈로 부각되었나요?"

raw_docs = base_retriever.invoke(question)
filtered_docs = filter_retriever.invoke(question)

print(f"필터 전: {len(raw_docs)}개 → 필터 후: {len(filtered_docs)}개")

print("\n[유지된 문서]")
for i, doc in enumerate(filtered_docs, start=1):
    print(f"\n[{i}] {doc_label(doc)}")
    print(doc.page_content[:500])

필터 전: 5개 → 필터 후: 5개

[유지된 문서]

[1] chunk_id=17 | doc_type=markdown | source_name=AI@Data_Report_CLEANED.md | source=data\AI@Data_Report_CLEANED.md
- - 클라우드 중심의 기술 적용이 스마트폰·개인 디바이스 등 온디바이스·엣지 환경으로 확장되는 경향이 나타남 - 이는 기술 발전이 성능 중심을 넘어 배포 환경 중심 다변화로 이동 하고 있음을 의미함 

- 현시점 AI 기술은 지능 구조 고도화와 성능 경쟁 가속이 동시에 심화되는 양상을 보이며, 클라우드 중심에서 온디바이스 등 다양한 활용 환경으로 확장되고 있음 

## **3.2.3 [AI 정책 토픽] - 안전성과 책임 중심의 규제 거버넌스 흐름** 

## **[ 그림 4 ] 안정성과 책임 중심의 규제 거버넌스 흐름 워드클라우드** 

AI 정책 워드클라우드는 안전성 확보, 규제·기준 정비, 책임·의무 강화가 핵심 이슈로 부각됨 안전, 위험, 방안, 시험: 안전성 확보와 위험 관리 중심 정책 

- - ‘안전’이 가장 주요하게 등장하며 AI 확산 속도 대비 위험 관리·안전 확보 체계를 시급히 강화해야 한다는 정책적 요구 가 높아지고 있음을 시사함 

- - ‘위험’, ‘시험’, 

[2] chunk_id=18 | doc_type=markdown | source_name=AI@Data_Report_CLEANED.md | source=data\AI@Data_Report_CLEANED.md
- - 가이드라인 중심의 자율 규제 단계에서 벗어나 법적 구속력 기반의 규제 집행 구조로 이행하는 흐름 을 반영함 

- 의무, 준수, 투명, 표시: 책임·의무 강화와 신뢰 기반 거버넌스 확립 

- - AI 개발자·기업·플랫폼에 요구되는 책임성과 준수 의무 강화가 정책적 핵심 이슈로 부상했음을 시사함 

- - 출력물 표시·데이터 출처 공개 등 투명성 강화를 통한 신뢰 기반 거버넌스 요구가 확대되는 흐름 으로 해석됨

## EmbeddingsFilter - 임베딩 유사도 기반 필터링

`EmbeddingsFilter`는 LLM이 아니라 임베딩 유사도를 기준으로 문서를 걸러낸다.

- 장점: LLM 호출이 적어 상대적으로 빠르고 비용이 낮다.
- 단점: 의미 판단이 단순하여 문맥적 관련성 판단은 LLM 기반 필터보다 약할 수 있다.

실습에서는 threshold 값을 조정하면서 필터링 결과가 어떻게 달라지는지 확인한다.

### similarity_threshold 해석

`similarity_threshold`는 질문과 문서의 임베딩 유사도가 이 값 이상일 때만 문서를 유지한다는 의미이다.

다만 이 값은 절대적인 기준이 아니다.  
사용하는 임베딩 모델, 문서 길이, VectorDB 설정에 따라 적절한 값이 달라질 수 있다.

실습에서는 0.70, 0.76, 0.82처럼 값을 바꿔 보면서 필터링 결과가 어떻게 달라지는지 확인할 수 있다.

In [15]:
from langchain_classic.retrievers.document_compressors import EmbeddingsFilter

# similarity_threshold: 이 값 이상인 문서만 유지한다.
# 값이 너무 높으면 관련 문서도 제거될 수 있고,
# 값이 너무 낮으면 관련 없는 문서가 많이 남을 수 있다.
emb_filter = EmbeddingsFilter(
    embeddings=embeddings,
    # similarity_threshold=0.76
    similarity_threshold=0.5
)

emb_filter_retriever = ContextualCompressionRetriever(
    base_compressor=emb_filter,
    base_retriever=base_retriever
)

question = "온디바이스 AI 확산은 어떤 의미인가요?"

raw_docs = base_retriever.invoke(question)
filtered_docs = emb_filter_retriever.invoke(question)

print(f"필터 전: {len(raw_docs)}개 → 필터 후: {len(filtered_docs)}개")

for i, doc in enumerate(filtered_docs, start=1):
    print(f"\n[{i}] {doc_label(doc)}")
    print(doc.page_content[:500])

필터 전: 5개 → 필터 후: 2개

[1] chunk_id=5 | doc_type=markdown | source_name=AI@Data_Report_CLEANED.md | source=data\AI@Data_Report_CLEANED.md
- - 생성형 AI 활용 영역 도 상담·요약을 넘어 기획·분석 등 고부가가치 업무로 확장되며, 기업 운영 방식 자체를 재정의하는 수준의 변화 를 촉발하고 있음 

- AI 기술은 모델·데이터 규모 경쟁을 넘어 합성데이터·멀티모달·에이전트형 AI 중심 으로 지능 구조 자체의 고도화가 진행되는 단계에 진입함 

- - 이처럼 AI가 산업·사회 전반으로 확산되며 이를 둘러싼 산업의 수요, 기술의 혁신, 정책의 대응이 상호 연계되는 복합적 변화 를 맞이함 

- 정책·사회 측면에서도 AI 활용 확산 속도가 빨라지며, 성장성과 함께 안전성·책임성 등 구조적 리스크 대응 필요성이 크게 증대되고 있음 

- - OECD AI Incidents Monitor에 따르면, 2010년대 후반 이후 AI 관련 사고·위험 보고 건수가 지속적으로 증가하고 있으며, 2023~2024년 이후 특히 가파른 상승 추세를 보이고 있음[2)] 

- 산업·기술·정책 간 변화 속도의 비대칭성으로 속도 격차 및 규제 지연

[2] chunk_id=17 | doc_type=markdown | source_name=AI@Data_Report_CLEANED.md | source=data\AI@Data_Report_CLEANED.md
- - 클라우드 중심의 기술 적용이 스마트폰·개인 디바이스 등 온디바이스·엣지 환경으로 확장되는 경향이 나타남 - 이는 기술 발전이 성능 중심을 넘어 배포 환경 중심 다변화로 이동 하고 있음을 의미함 

- 현시점 AI 기술은 지능 구조 고도화와 성능 경쟁 가속이 동시에 심화되는 양상을 보이며, 클라우드 중심에서 온디바이스 등 다양한 활용 환경으로 확장되고 있음 

## **3.2.3 [AI 정책 토픽] - 안전성과 책임 중심의 규제 거

# [선택 실습]
- 실행 시간이 길 수 있으므로 수업 환경이 준비된 경우에만 실행하세요.

## 선택 실습: Pipeline - Reranking + Compression 결합

`DocumentCompressorPipeline`을 사용하면 여러 압축 단계를 순서대로 결합할 수 있다.

이번 실습에서는 다음 흐름을 사용한다.

```text
Vector Store에서 후보 문서 검색
  ↓
CrossEncoderReranker로 관련성 높은 문서만 선별
  ↓
LLMChainExtractor로 질문 관련 문장만 추출
  ↓
압축된 context 생성
```

이 방식은 검색 결과가 많거나 청크 안에 여러 주제가 섞여 있을 때 유용하다.  

추가 설치 라이브러리
```
uv add sentence-transformers torch transformers accelerate
```

In [11]:
from langchain_classic.retrievers import ContextualCompressionRetriever
from langchain_classic.retrievers.document_compressors import (
    DocumentCompressorPipeline,
    CrossEncoderReranker,
    LLMChainExtractor,
)
from langchain_community.cross_encoders import HuggingFaceCrossEncoder

# Reranking + Compression 2단계 파이프라인
# 1단계 Reranking : BGE 리랭커로 k=10 후보 중 관련성 상위 5개로 압축한다.
# 2단계 Compression: LLMChainExtractor로 각 문서에서 질문과 관련된 문장만 추출한다.
pipeline = DocumentCompressorPipeline(
    transformers=[
        CrossEncoderReranker(
            model=HuggingFaceCrossEncoder(model_name="BAAI/bge-reranker-v2-m3"),
            top_n=5
        ),
        LLMChainExtractor.from_llm(llm),
    ]
)

pipeline_retriever = ContextualCompressionRetriever(
    base_compressor=pipeline,
    base_retriever=vector_store.as_retriever(search_kwargs={"k": 10})
)

question = "AI 산업, AI 기술, AI 정책의 핵심 이슈는 각각 어떻게 다른가요?"

raw_docs = vector_store.as_retriever(search_kwargs={"k": 10}).invoke(question)
results = pipeline_retriever.invoke(question)

print(f"검색: {len(raw_docs)}개 → 리랭킹+압축 후: {len(results)}개")

for i, doc in enumerate(results, start=1):
    print(f"\n[{i}] {len(doc.page_content)}자 | {doc_label(doc)}")
    print(doc.page_content[:700])

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

검색: 10개 → 리랭킹+압축 후: 5개

[1] 564자 | chunk_id=19 | doc_type=markdown | source_name=AI@Data_Report_CLEANED.md | source=data\AI@Data_Report_CLEANED.md
## **3.2.4 [AI 산업·기술·정책 토픽] - 종합 결과 및 시사점** 

|**분야**|**토픽명**|**주요 키워드**|**핵심 이슈**|
|---|---|---|---|
|AI 산업|시장 규모
확대와 비즈니스
구조 전환|도입, 확대, 성장세,
확산. 규모, 성장,
글로벌, 분기, 비용,
자금, 인프라, 센터,
효율, 에이전트|- AI 도입·확산 가속에 따른 산업 구조 변화
- 산업 규모 확대 및 글로벌 경쟁 심화
- 인프라·에이전트 기반 운영 모델 변화|
|AI 기술|기능 고도화와
적용 범위 확장|멀티, 추론, 기술,
기능. 개발, 강화,
향상, 성능, 디바이스,
서비스, 활용, 사례|- 지능 구조 고도화 중심의 기술 발전 흐름
- 성능 고도화와 개발·업데이트 주기의 가속
- 기술 적용 환경 다변화 및 온디바이스 확산|
|AI 정책|안전성과 책임
중심의 규제
거버넌스|안전, 위험, 방안,
시험, 규제, 기본법,
시행, 기준, 의무,
준수, 투명, 표시|- 안전성 확보와 위험 관리 중심 정책
- 규제·제도 정비 중심의 법적 틀 구축 흐름
- 책임·의무 강화와 신뢰 기반 거버넌스 확립|

[2] 121자 | chunk_id=18 | doc_type=markdown | source_name=AI@Data_Report_CLEANED.md | source=data\AI@Data_Report_CLEANED.md
- AI 개발자·기업·플랫폼에 요구되는 책임성과 준수 의무 강화가 정책적 핵심 이슈로 부상했음을 시사함 

- 현시점 AI 정책 담론은 전체적으로 ‘신뢰 기반 AI 거버넌스’ 구축 이 정책 영역의 핵심 방향으로 나타남

[3] 418자 | chunk_id=6 | doc_type=markdown

## Compression + RAG 체인

이제 압축된 검색 결과를 사용해 최종 RAG 답변을 생성한다.

일반 RAG는 검색된 청크 전체를 context로 넣지만,  
Contextual Compression RAG는 질문과 관련된 내용만 줄여 넣는다는 차이가 있다.

In [12]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

answer_prompt = ChatPromptTemplate.from_messages([
    ("system", """아래 문서에 근거해서만 질문에 답하세요.
문서에서 확인할 수 없는 내용은 '문서에서 확인할 수 없습니다'라고 답하세요.
한 줄에 한 문장씩 정리하세요.

[문서]
{context}"""),
    ("human", "{question}")
])

### Compression + RAG 체인 실행

In [13]:
# LLMChainExtractor 기반 Compression → 질문 답변 체인
compression_rag_chain = (
    {
        "context": extractor_retriever  | format_docs,
        "question": RunnablePassthrough()
    }
    | answer_prompt
    | llm
    | StrOutputParser()
)

question = "2026년 AI 기술 전망에서 핵심 변화는 무엇인가요?"

answer = compression_rag_chain.invoke(question)

print(answer)

2026년 AI 기술 전망에서 핵심 변화는 지능 구조 고도화와 데이터 한계 보완을 중심으로 진화할 것으로 예상됨. 

합성데이터, 추론형 AI, 멀티모달 기술이 주요 경쟁 축으로 자리 잡을 것으로 보임. 

이러한 기술 고도화는 입력 유형과 데이터 범위를 확장하여 AI의 활용 가능성을 넓히고, 복잡한 상황에서의 판단 능력을 강화하는 방향으로 진화할 전망임. 

고품질 데이터 생성, 멀티모달, 고급 추론 기술이 결합되어 AI의 상황 이해와 문제 해결 능력이 향상될 것으로 기대됨. 

서비스와 산업 전반의 활용도도 확대될 전망임.


### 선택 실습: Pipeline 기반 RAG 체인

앞의 기본 Compression RAG 체인은 `LLMChainExtractor`만 사용한다.  
아래 체인은 선택 실습으로, `Reranking + LLMChainExtractor`를 결합한 `pipeline_retriever`를 사용한다.

단, Hugging Face Cross-Encoder 모델 다운로드가 필요하므로 실행 환경에 따라 시간이 오래 걸릴 수 있다.

In [14]:
# Pipeline으로 Reranking + Compression → 질문 답변 체인
pipeline_rag_chain = (
    {
        "context": pipeline_retriever | format_docs,
        "question": RunnablePassthrough()
    }
    | answer_prompt
    | llm
    | StrOutputParser()
)

question = "2026년 AI 기술 전망에서 핵심 변화는 무엇인가요?"

answer = pipeline_rag_chain.invoke(question)

print(answer)

2026년 AI 기술 전망에서 핵심 변화는 지능 구조 고도화와 데이터 한계 보완입니다.  
합성데이터, 추론형 AI, 멀티모달 기술이 주요 경쟁 축으로 자리 잡을 것입니다.  
기술 고도화는 입력 유형과 데이터 범위를 확장하여 AI의 활용 가능성을 넓힐 것입니다.  
고품질 데이터 생성, 멀티모달, 고급 추론 기술이 결합되어 AI의 상황 이해와 문제 해결 능력이 향상될 것입니다.  
내부 알고리즘의 고도화와 기술 적용의 확대가 나타날 것입니다.  
안전 관리 중심의 제도 설계가 빠르게 진전될 것입니다.  
산업, 기술, 정책의 변화가 동시에 심화되며 AI 생태계의 구조 재편이 가속화될 가능성이 있습니다.


## 정리

| Compressor | 방식 | 적합 상황 |
|---|---|---|
| LLMChainExtractor | LLM이 질문과 관련된 내용만 추출 | 청크 안에 여러 주제가 섞여 있는 경우 |
| LLMChainFilter | LLM이 문서 전체의 관련성 판단 | 관련 없는 문서를 제거해야 하는 경우 |
| EmbeddingsFilter | 임베딩 유사도 임계값으로 필터링 | 빠른 필터링과 비용 절감이 필요한 경우 |
| Pipeline | Reranking과 Compression을 결합 | 검색 정밀도와 context 품질을 함께 높이고 싶은 경우 |

## 이번 실습의 핵심

```text
AI 트렌드 보고서 Vector Store
  ↓
기본 검색
  ↓
Contextual Compression
  ↓
질문과 관련된 내용 중심의 context 생성
  ↓
LLM 답변 생성
```

Contextual Compression은 단순히 검색된 문서의 양을 줄이는 것이 목적이 아니다.  
핵심은 **질문과 관련 없는 노이즈를 줄여 답변에 사용할 context 품질을 높이는 것**이다.

다음 단계: **출처 기반 답변 생성**